# IA | Proiect RPC #2
- Nume: Irimia David
- Grupa: 241
- Joc: Hex

In [33]:
import time, pygame, sys, copy, math

Limita maxima de explorare in arborele de stari (performanta)

In [34]:
ADANCIME_MAX = 3

Varfurile unui hexagon

In [35]:
def calculeaza_varfuri_hexagon(centru_x, centru_y, raza):
    varfuri = []
    for i in range(6):
        unghi = math.pi / 3 * i + math.pi / 6 # adaugam math.pi/6 pentru a roti hexagonul cu varful in sus
        x = centru_x + raza * math.cos(unghi)
        y = centru_y + raza * math.sin(unghi)
        varfuri.append((x, y))
    return varfuri

Clasa jocului Hex

In [36]:
class InfoJoc:
    dimensiune_tabla=7
    jucator = None
    calculator = None
    GOL='#'
    RAZA_HEXAGON = 25

    @classmethod
    def initializeaza(cls, display):
        cls.display = display
        distanta_orizontala = math.sqrt(3) * cls.RAZA_HEXAGON
        distanta_verticala = 1.5 * cls.RAZA_HEXAGON
        padding_orizontal = 140
        padding_vertical = 100
        cls.grid_extins = {}
        for linie in range(-1, cls.dimensiune_tabla + 1):
            for coloana in range(-1, cls.dimensiune_tabla + 1):
                cx = padding_orizontal + coloana * distanta_orizontala + linie * (distanta_orizontala / 2)
                cy = padding_vertical + linie * distanta_verticala
                cls.grid_extins[(linie, coloana)] = (cx, cy)
        cls.celuleGrid = []
        for linie in range(cls.dimensiune_tabla):
            rand_curent = []
            for coloana in range(cls.dimensiune_tabla):
                rand_curent.append(cls.grid_extins[(linie, coloana)])
            cls.celuleGrid.append(rand_curent)

    def deseneaza_grid(self, marcaj=None):
        CULOARE_GOL = (50, 50, 50) # gri inchis
        CULOARE_JUCATOR = (255, 50, 50) # rosu
        CULOARE_CALCULATOR = (50, 50, 255) # albastru
        CULOARE_BORDURA = (255, 255, 255) # alb
        self.__class__.display.fill((0, 0, 0)) # negru
        
        # Hexagoanele din margine
        for linie in range(-1, self.dimensiune_tabla + 1):
            for coloana in range(-1, self.dimensiune_tabla + 1):
                este_margine = (linie == -1 or linie == self.dimensiune_tabla or coloana == -1 or coloana == self.dimensiune_tabla)
                if este_margine:
                    cx, cy = self.__class__.grid_extins[(linie, coloana)]
                    varfuri = calculeaza_varfuri_hexagon(cx, cy, self.__class__.RAZA_HEXAGON)
                    if linie == -1 or linie == self.dimensiune_tabla:
                        culoare_mal = CULOARE_JUCATOR
                    else:
                        culoare_mal = CULOARE_CALCULATOR
                    pygame.draw.polygon(self.__class__.display, culoare_mal, varfuri)

        # Hexagoanele din interior
        for linie in range(self.dimensiune_tabla):
            for coloana in range(self.dimensiune_tabla):
                cx, cy = self.__class__.celuleGrid[linie][coloana]
                varfuri = calculeaza_varfuri_hexagon(cx, cy, self.__class__.RAZA_HEXAGON)
                stare_celula = self.matr[linie][coloana]
                
                if stare_celula == self.__class__.jucator:
                    culoare = CULOARE_JUCATOR
                elif stare_celula == self.__class__.calculator:
                    culoare = CULOARE_CALCULATOR
                else:
                    culoare = CULOARE_GOL
                    
                pygame.draw.polygon(self.__class__.display, culoare, varfuri)
                pygame.draw.polygon(self.__class__.display, CULOARE_BORDURA, varfuri, 1)

        pygame.display.update()

    def __init__(self, tabla=None):
        if tabla:
            self.matr=tabla
        else:
            self.matr= [[InfoJoc.GOL]* InfoJoc.dimensiune_tabla  for _ in range(InfoJoc.dimensiune_tabla) ]

    @classmethod
    def jucator_opus(cls, jucator):
        return 'a' if jucator=='r' else 'r'

    def final(self):
        def obtine_vecini(l, c):
            return [(l, c-1), (l, c+1), (l-1, c), (l-1, c+1), (l+1, c-1), (l+1, c)]

        ############
        # DFS Rosu #
        ############

        vizitat_rosu = set()
        def dfs_rosu(l, c):
            if l == self.dimensiune_tabla - 1:
                return True
            
            vizitat_rosu.add((l, c))

            for vecin_l, vecin_c in obtine_vecini(l, c):
                if 0 <= vecin_l < self.dimensiune_tabla and 0 <= vecin_c < self.dimensiune_tabla:
                    if (vecin_l, vecin_c) not in vizitat_rosu and self.matr[vecin_l][vecin_c] == 'r':
                        if dfs_rosu(vecin_l, vecin_c):
                            return True
            return False
    
        # Porneste DFS pentru fiecare piesa rosie de pe primul rand din tabla de joc
        for c in range(self.dimensiune_tabla):
            if self.matr[0][c] == 'r' and (0, c) not in vizitat_rosu:
                if dfs_rosu(0, c):
                    return 'r'
                
        ################
        # DFS Albastru #
        ################

        vizitat_albastru = set()
        def dfs_albastru(l, c):
            if c == self.dimensiune_tabla - 1:
                return True
            
            vizitat_albastru.add((l, c))

            for vl, vc in obtine_vecini(l, c):
                if 0 <= vl < self.dimensiune_tabla and 0 <= vc < self.dimensiune_tabla:
                    if (vl, vc) not in vizitat_albastru and self.matr[vl][vc] == 'a':
                        if dfs_albastru(vl, vc):
                            return True
            return False
    
        # Porneste DFS pentru fiecare piesa albstra de pe prima coloana din tabla de joc
        for l in range(self.dimensiune_tabla):
            if self.matr[l][0] == 'a' and (l, 0) not in vizitat_albastru:
                if dfs_albastru(l, 0):
                    return 'a'

        ###############
        # Tabla plina #
        ###############

        remiza = True
        for linie in self.matr:
            for element in linie:
                if element == InfoJoc.GOL:
                    remiza = False
        if remiza:
            return "remiza"
        return False

    def mutari(self, jucator): # jucator = culoarea jucatorului care muta
        lMutari=[]
        for i in range(InfoJoc.dimensiune_tabla):
            for j in range(InfoJoc.dimensiune_tabla):
                if self.matr[i][j] == InfoJoc.GOL:
                    mutareNoua = copy.deepcopy(self.matr)
                    mutareNoua[i][j] = jucator
                    lMutari.append(InfoJoc(mutareNoua)) # nu pun doar tabla, vreau toate metodele din InfoJoc     
        return lMutari
                
    def estimeaza_scor(self, restAdancime):
        scor_final = self.final()
        if scor_final == self.__class__.calculator:
            return 1000 + restAdancime
        elif scor_final == self.__class__.jucator:
            return -1000 - restAdancime
        elif scor_final == "remiza":
            return 0

        scor = 0
        centru = self.dimensiune_tabla // 2

        def numar_vecini_aliati(l, c, culoare):
            vecini = [(l, c-1), (l, c+1), (l-1, c), (l-1, c+1), (l+1, c-1), (l+1, c)]
            nr = 0
            for vl, vc in vecini:
                if 0 <= vl < self.dimensiune_tabla and 0 <= vc < self.dimensiune_tabla:
                    if self.matr[vl][vc] == culoare:
                        nr += 1
            return nr
        for linie in range(self.dimensiune_tabla):
            for coloana in range(self.dimensiune_tabla):
                piesa = self.matr[linie][coloana]
                if piesa == self.__class__.calculator: # 'a'
                    scor += 15
                    distanta_stanga = coloana
                    distanta_dreapta = (self.dimensiune_tabla - 1) - coloana
                    scor += (self.dimensiune_tabla - abs(distanta_stanga - distanta_dreapta)) * 3
                    aliati = numar_vecini_aliati(linie, coloana, 'a')
                    scor += aliati * 8
                    if abs(linie - centru) <= 1 and abs(coloana - centru) <= 1:
                        scor += 10
                elif piesa == self.__class__.jucator: # 'r'
                    scor -= 15
                    distanta_sus = linie
                    distanta_jos = (self.dimensiune_tabla - 1) - linie
                    scor -= (self.dimensiune_tabla - abs(distanta_sus - distanta_jos)) * 3
                    aliati_om = numar_vecini_aliati(linie, coloana, 'r')
                    scor -= aliati_om * 8
                    if abs(linie - centru) <= 1 and abs(coloana - centru) <= 1:
                        scor -= 10
        return scor
    
    def sirAfisare(self):
        sir="  |"
        sir+=" ".join([str(i) for i in range(self.dimensiune_tabla)])+"\n"
        sir+="-"*(self.dimensiune_tabla+1)*2+"\n"
        for i in range(self.dimensiune_tabla):
                sir+= str(i)+" |"+" ".join([str(x) for x in self.matr[i]])+"\n"
        return sir

    def __str__(self):
        return self.sirAfisare()

    def __repr__(self):
        return self.sirAfisare() 
    

Clasa starilor de joc

In [ ]:
class Stare:
    """
    Functioneaza cu conditia ca in cadrul clasei InfoJoc sa fie definiti 
        - jucator, calculator (i.e. cei doi jucatori)
    De asemenea cere ca in clasa InfoJoc sa fie definita si o metoda numita
        - mutari() care ofera lista cu configuratiile posibile in urma mutarii unui jucator
    """
    def __init__(self, tabla_joc, j_curent, restAdancime, parinte=None, estimare=None):
        self.tabla_joc=tabla_joc
        self.j_curent=j_curent
        self.restAdancime=restAdancime    
        self.estimare=estimare
        self.mutari_posibile=[]
        self.stare_aleasa=None

    def mutari(self):        
        l_mutari=self.tabla_joc.mutari(self.j_curent)
        juc_opus=InfoJoc.jucator_opus(self.j_curent)
        l_stari_mutari=[Stare(mutare, juc_opus, self.restAdancime-1, parinte=self) for mutare in l_mutari]

        return l_stari_mutari
        
    def __str__(self):
        sir= str(self.tabla_joc) + "(Jucator curent: " + self.j_curent+")\n"
        return sir

Algoritmul Alpha-Beta

In [38]:
def alpha_beta(alpha, beta, stare):
    if stare.tabla_joc.final() or stare.restAdancime==0:
        stare.estimare= stare.tabla_joc.estimeaza_scor(stare.restAdancime)
        return stare
    
    if stare.j_curent==InfoJoc.calculator:
        stare.estimare=float("-inf")
        for mutare in stare.mutari():
            mutare = alpha_beta(alpha, beta, mutare) # ii dau nodul si alfa beta vine cu arborele si scorul
            if mutare.estimare > alpha:
                alpha = mutare.estimare
                stare.estimare = mutare.estimare
                if alpha >= beta:
                    return stare
                stare.stare_aleasa = mutare
    else:
        stare.estimare=float("+inf")
        for mutare in stare.mutari():
            mutare = alpha_beta(alpha, beta, mutare) # ii dau nodul si alfa beta vine cu arborele si scorul
            if mutare.estimare < beta:
                beta = mutare.estimare
                stare.estimare = mutare.estimare
                if alpha >= beta:
                    return stare
                stare.stare_aleasa = mutare

    return stare

Mesaj final de joc

In [ ]:
def afis_daca_final(stare_curenta):
    final = stare_curenta.tabla_joc.final()
    if final:
        if final == "remiza":
            print("Remiza!")
        else:z
            print("A castigat " + final)          
        return True
    return False

Functia principala

In [40]:
def main():
    # Initializeaza tabla
    tabla_curenta=InfoJoc()
    print("Tabla initiala")
    print(str(tabla_curenta))
    InfoJoc.calculator = 'a'
    InfoJoc.jucator = 'r'
    
    # Creeaza starea initiala
    stare_curenta = Stare(tabla_curenta, 'r', ADANCIME_MAX)
    
    # Setari interfata grafica
    pygame.init()
    pygame.display.set_caption('Jocul Hex')
    ecran = pygame.display.set_mode(size=(100 + InfoJoc.dimensiune_tabla * 100, 100 + InfoJoc.dimensiune_tabla * 50))
    InfoJoc.initializeaza(ecran)
    stare_curenta.tabla_joc.deseneaza_grid()
    pygame.display.flip()   
    este_prima_mutare_calculator = True
    coordonate_prima_piesa_jucator = None
    while True :
        ####################
        # Tura jucatorului #
        ####################
        if stare_curenta.j_curent == InfoJoc.jucator:
            for event in pygame.event.get():
                if event.type== pygame.QUIT:
                    pygame.quit() # Inchide fereastra
                    sys.exit()
                elif event.type == pygame.MOUSEBUTTONDOWN: # Click
                    pos = pygame.mouse.get_pos() # Coordonatele clickului
                    mutare_valida = False
                    for linie in range(InfoJoc.dimensiune_tabla):
                        for coloana in range(InfoJoc.dimensiune_tabla):
                            cx, cy = InfoJoc.celuleGrid[linie][coloana]
                            if math.hypot(pos[0] - cx, pos[1] - cy) <= InfoJoc.RAZA_HEXAGON: # verifica daca click-ul a fost in hexagonul curent
                                if stare_curenta.tabla_joc.matr[linie][coloana] == InfoJoc.GOL: # verifica daca este libera
                                    stare_curenta.tabla_joc.matr[linie][coloana] = InfoJoc.jucator
                                    stare_curenta.tabla_joc.deseneaza_grid()
                                    mutare_valida = True
                                    if coordonate_prima_piesa_jucator is None:
                                        coordonate_prima_piesa_jucator = (linie, coloana)
                                    print("\nTabla dupa mutarea jucatorului")
                                    print(str(stare_curenta))
                                    if afis_daca_final(stare_curenta):
                                        return
                                    stare_curenta.j_curent = InfoJoc.jucator_opus(stare_curenta.j_curent)
                        if mutare_valida:
                            break
        #######################
        # Tura calculatorului #
        #######################
        else:
            timp_inainte = int(round(time.time() * 1000))
            if este_prima_mutare_calculator and coordonate_prima_piesa_jucator is not None:
                este_prima_mutare_calculator = False
                linie, coloana = coordonate_prima_piesa_jucator
                centru = InfoJoc.dimensiune_tabla // 2
                if abs(linie - centru) <= 1 and abs(coloana - centru) <= 1:
                    print("Calculatorul a decis sa schimbe prima piesa in albastru")
                    stare_curenta.tabla_joc.matr[linie][coloana] = InfoJoc.calculator
                    stare_curenta.tabla_joc.deseneaza_grid()
                    stare_curenta.j_curent = InfoJoc.jucator_opus(stare_curenta.j_curent)
                    continue
            stare_actualizata = alpha_beta(float("-inf"), float("+inf"), stare_curenta)
            stare_curenta.tabla_joc = stare_actualizata.stare_aleasa.tabla_joc

            print("\nTabla dupa mutarea calculatorului")
            print(str(stare_curenta))

            stare_curenta.tabla_joc.deseneaza_grid()
            timp_dupa = int(round(time.time() * 1000))

            print(f"\nCalculatorul a gandit timp de {timp_dupa - timp_inainte} milisecunde")

            if afis_daca_final(stare_curenta):
                break

            stare_curenta.j_curent = InfoJoc.jucator_opus(stare_curenta.j_curent)

if __name__ == "__main__" :
    main()
    while True :
        for event in pygame.event.get():
            if event.type== pygame.QUIT:
                pygame.quit()
                sys.exit()

Tabla initiala
  |0 1 2 3 4 5 6
----------------
0 |# # # # # # #
1 |# # # # # # #
2 |# # # # # # #
3 |# # # # # # #
4 |# # # # # # #
5 |# # # # # # #
6 |# # # # # # #


Tabla dupa mutarea jucatorului
  |0 1 2 3 4 5 6
----------------
0 |# # # # # # #
1 |# # # # # # #
2 |# # # # # # #
3 |# # # # # # #
4 |# # # # # # #
5 |# # # # # # #
6 |# # # # r # #
(Jucator curent: r)


Tabla dupa mutarea calculatorului
  |0 1 2 3 4 5 6
----------------
0 |# # # # # # #
1 |# # # # # # #
2 |# # # # # # #
3 |# # # a # # #
4 |# # # # # # #
5 |# # # # # # #
6 |# # # # r # #
(Jucator curent: a)


Calculatorul a gandit timp de 1803 milisecunde

Tabla dupa mutarea jucatorului
  |0 1 2 3 4 5 6
----------------
0 |# # # # # # #
1 |# # # # # # #
2 |# # # # # # #
3 |# # # a r # #
4 |# # # # # # #
5 |# # # # # # #
6 |# # # # r # #
(Jucator curent: r)


Tabla dupa mutarea calculatorului
  |0 1 2 3 4 5 6
----------------
0 |# # # # # # #
1 |# # # # # # #
2 |# # # a # # #
3 |# # # a r # #
4 |# # # # # # #
5 |# # #

SystemExit: 

/home/indavid04/fmi/repos/.venv/lib64/python3.13/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
